In [24]:
from tensorflow.keras.applications import EfficientNetV2B3
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense,Dropout,GlobalAveragePooling2D
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report,confusion_matrix
import matplotlib.pyplot as plt
import clahe

In [25]:
training_path = "../Datasets/data/train/Medical/X-Ray/humerus/Train"
testining_path = "../Datasets/data/test/Medical/X-Ray/humerus/Test"
validation_path = "../Datasets/data/vaild/Medical/X-Ray/humerus"

In [26]:
train_aug = ImageDataGenerator(
    preprocessing_function = lambda img : clahe.clahe_preprocess(img,preprocess_input),
    rotation_range = 10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    shear_range=0.05,
    validation_split = 0.2
)
valid_aug = ImageDataGenerator(
    preprocessing_function=lambda img: clahe.clahe_preprocess(img, preprocess_input),
    validation_split=0.2
)
test_aug = ImageDataGenerator(preprocessing_function = lambda img : clahe.clahe_preprocess(img,preprocess_input))

In [27]:
train_set = train_aug.flow_from_directory(
    training_path,
    target_size = (224,224),
    batch_size=8,
    class_mode="binary",
    seed=42
)
validation_set = train_aug.flow_from_directory(
    validation_path,
    target_size=(224, 224),
    batch_size=8,
    class_mode="binary",
    subset="validation",
    seed=42,
)
test_set = test_aug.flow_from_directory(
    testining_path,
    target_size = (224,224),
    batch_size=8,
    class_mode="binary",
    shuffle = False
)

Found 1440 images belonging to 2 classes.
Found 40 images belonging to 2 classes.
Found 538 images belonging to 2 classes.


In [28]:
base_model = EfficientNetV2B3(weights="imagenet",include_top=False,input_shape=(224,224,3),pooling = 'avg')

In [29]:
base_model.trainable = False

In [30]:
x = base_model.output
x = Dense(256,activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(512,activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256,activation='relu')(x)
output = Dense(1,activation='sigmoid')(x)

In [31]:
model = Model(inputs = base_model.input, outputs = output)

In [32]:
model.compile(optimizer='adam',metrics=['accuracy'],loss="binary_crossentropy")

In [33]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


In [34]:
model.fit(train_set,validation_data=validation_set,epochs=40,callbacks=early_stop)

Epoch 1/40


I0000 00:00:1789145848.153491   16071 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_139140__.393


180/180 ━━━━━━━━━━━━━━━━━━━━ 55s 129ms/step - accuracy: 0.6542 - loss: 0.6544 - val_accuracy: 0.8000 - val_loss: 0.4884
Epoch 2/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 14s 78ms/step - accuracy: 0.7486 - loss: 0.5477 - val_accuracy: 0.8000 - val_loss: 0.4159
Epoch 3/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 85ms/step - accuracy: 0.7792 - loss: 0.5107 - val_accuracy: 0.8000 - val_loss: 0.4310
Epoch 4/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 13s 72ms/step - accuracy: 0.7646 - loss: 0.4967 - val_accuracy: 0.7750 - val_loss: 0.3899
Epoch 5/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 85ms/step - accuracy: 0.7681 - loss: 0.5086 - val_accuracy: 0.8250 - val_loss: 0.4462
Epoch 6/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 13s 72ms/step - accuracy: 0.7917 - loss: 0.4533 - val_accuracy: 0.8750 - val_loss: 0.3213
Epoch 7/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.8174 - loss: 0.4432 - val_accuracy: 0.8000 - val_loss: 0.3822
Epoch 8/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 84ms/step - accuracy: 0.8069 - loss: 0.4460 - val_accura

In [35]:
prediction_prob = model.predict(test_set)

68/68 ━━━━━━━━━━━━━━━━━━━━ 16s 139ms/step


In [36]:
prediction = (prediction_prob >= 0.5).astype(int)

In [37]:
true_val = test_set.classes


In [38]:
print(classification_report(true_val,prediction))
print(confusion_matrix(true_val,prediction))

              precision    recall  f1-score   support

           0       0.87      0.69      0.77       270
           1       0.74      0.90      0.81       268

    accuracy                           0.79       538
   macro avg       0.80      0.79      0.79       538
weighted avg       0.81      0.79      0.79       538

[[186  84]
 [ 28 240]]


In [39]:
model1 = model

In [40]:
base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False

In [41]:
model.compile(optimizer=Adam(1e-5),loss="binary_crossentropy",metrics=['accuracy'])

In [42]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7
)

In [43]:
model.fit(train_set,validation_data=validation_set,epochs=30,callbacks=[ early_stop, reduce_lr])

Epoch 1/30


I0000 00:00:1789146070.977877   16075 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_199069__.425


180/180 ━━━━━━━━━━━━━━━━━━━━ 58s 126ms/step - accuracy: 0.7778 - loss: 0.4971 - val_accuracy: 0.8500 - val_loss: 0.3671 - learning_rate: 1.0000e-05
Epoch 2/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 11s 63ms/step - accuracy: 0.7951 - loss: 0.4479 - val_accuracy: 0.8250 - val_loss: 0.4127 - learning_rate: 1.0000e-05
Epoch 3/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 16s 89ms/step - accuracy: 0.7944 - loss: 0.4382 - val_accuracy: 0.8750 - val_loss: 0.3288 - learning_rate: 1.0000e-05
Epoch 4/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 14s 75ms/step - accuracy: 0.8083 - loss: 0.4324 - val_accuracy: 0.8000 - val_loss: 0.3946 - learning_rate: 1.0000e-05
Epoch 5/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 85ms/step - accuracy: 0.8160 - loss: 0.4252 - val_accuracy: 0.8250 - val_loss: 0.4249 - learning_rate: 1.0000e-05
Epoch 6/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 84ms/step - accuracy: 0.8229 - loss: 0.4056 - val_accuracy: 0.8500 - val_loss: 0.3614 - learning_rate: 2.0000e-06
Epoch 7/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 85ms/step - accura

In [44]:
prob = model.predict(test_set)

68/68 ━━━━━━━━━━━━━━━━━━━━ 15s 152ms/step


In [45]:
pre = (prob >= 0.5).astype(int)

In [46]:
print(classification_report(true_val,pre))
print(confusion_matrix(true_val,pre))

              precision    recall  f1-score   support

           0       0.85      0.76      0.80       270
           1       0.78      0.86      0.82       268

    accuracy                           0.81       538
   macro avg       0.81      0.81      0.81       538
weighted avg       0.81      0.81      0.81       538

[[204  66]
 [ 37 231]]


In [47]:
model.save("humerus_model.keras")